# Hathor AI — train your own LSTM music model from scratch

This notebook trains **a new, randomly initialized LSTM** on MIDI note patterns. It does not download or fine-tune a pretrained music-generation model. The trained network predicts a polyphonic piano-roll (active notes over time), which is converted to MIDI and rendered into MP3 with FluidSynth.

**What you will export:** `hathor_lstm_weights.npz` (custom learned LSTM weights + genre labels + small seed contexts), `hathor_lstm.keras` (the Keras training model), sample MIDI/MP3, and metadata. The website loads the NPZ file using NumPy—TensorFlow is only needed in this Colab notebook for training.

Important limitations:
- The default MAESTRO corpus is classical solo-piano MIDI. It gives you a real starting point, but it does **not** teach the model jazz, Indian music, or instrument-specific performance unless you add legally usable MIDI files for those styles.
- The dropdown exposes only genres that actually appeared in the training folders. To add a style, add MIDI files under a folder named for that style and retrain.
- Selecting Flute/Violin/Sitar changes the SoundFont instrument used when rendering; the initial MAESTRO model learns note patterns from piano performances, not flute technique. A standard SoundFont is an instrument sample library, not a pretrained music-generation model.
- LSTM MIDI generation is a research-quality starting point, not a guarantee of studio-mastered audio. Listen to and evaluate results before presenting this as production quality.

## Dataset licensing
MAESTRO's MIDI-only archive is small enough for a Colab workflow, but MAESTRO is **CC BY-NC-SA 4.0**. Use it for non-commercial experimentation only unless you have confirmed the licence permits your use. For commercial use, train with MIDI that you created or data whose licence explicitly allows your intended use. Do not upload copyrighted songs without permission.

## 1. Select a GPU runtime

In Colab on Android, open **Runtime → Change runtime type → T4 GPU** if available. GPU access can vary. CPU training is possible but slower.

In [ ]:
!pip -q install pretty_midi music21 tensorflow matplotlib requests

import os, json, math, random, zipfile, time, subprocess, shutil
from collections import defaultdict
from pathlib import Path
import numpy as np
import pretty_midi
import tensorflow as tf
import requests
import matplotlib.pyplot as plt
from music21 import converter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Hathor LSTM workspace ready.")

## 2. Download MIDI training data

The default is the **MIDI-only MAESTRO v3.0.0 archive** (about 56 MB). The notebook downloads MIDI files only—not the 100+ GB audio archive. Each file is treated as `classical` unless its folder is named for another supported genre.

In [ ]:
DATA_DIR = Path("/content/hathor_dataset")
CLASSICAL_DIR = DATA_DIR / "classical"
ZIP_PATH = Path("/content/maestro-v3.0.0-midi.zip")
DATA_URL = "https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CLASSICAL_DIR.mkdir(parents=True, exist_ok=True)


def safe_extract(zip_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path inside zip: {member.filename}")
        archive.extractall(destination)

if not any(CLASSICAL_DIR.rglob("*.mid")) and not any(CLASSICAL_DIR.rglob("*.midi")):
    if not ZIP_PATH.exists():
        print("Downloading official MIDI-only MAESTRO archive...")
        with requests.get(DATA_URL, stream=True, timeout=120) as response:
            response.raise_for_status()
            with ZIP_PATH.open("wb") as out:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        out.write(chunk)
    safe_extract(ZIP_PATH, CLASSICAL_DIR)

midi_files = sorted([p for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".mid", ".midi"}])
print(f"Found {len(midi_files):,} MIDI files under {DATA_DIR}")
if not midi_files:
    raise RuntimeError("No MIDI files were found. Check the dataset download or upload path.")

## 3. Add more genres (optional)

The default MAESTRO files only support `Classical`. To teach Jazz, Pop, Indian Fusion, Ambient, etc.:

1. Prepare a ZIP named `hathor_extra_midi.zip` on your phone.
2. Put each style in its own top-level folder inside the ZIP, for example:
   - `jazz/song_001.mid`
   - `jazz/song_002.mid`
   - `ambient/ambient_001.mid`
   - `indian_fusion/theme_001.mid`
3. In Colab's left **Files** panel, upload that ZIP to `/content/hathor_extra_midi.zip` **before running this cell**.
4. Run the cell. It extracts the ZIP into the training folder.

Use only data you have permission to train on. Folder names supported by the notebook include `classical`, `jazz`, `pop`, `rock`, `electronic`, `ambient`, `cinematic`, `orchestral`, `acoustic`, `indian_fusion`, `indian_classical`, `folk`, `blues`, `world_music`, `lo_fi`, `neo_soul`, and `fantasy`. This cell is safe to skip if you only want the default Classical model.

In [ ]:
EXTRA_ZIP = Path("/content/hathor_extra_midi.zip")
if EXTRA_ZIP.is_file():
    print("Extracting additional labelled MIDI folders...")
    safe_extract(EXTRA_ZIP, DATA_DIR)
    print("Extra MIDI files loaded.")
else:
    print("No additional ZIP found; using the default Classical MAESTRO corpus.")

midi_files = sorted([p for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".mid", ".midi"}])
print(f"Total MIDI files available: {len(midi_files):,}")

## 4. Inspect a score with `music21`

`music21` lets us inspect a MIDI as musical note/chord objects. The faster numeric preprocessing below uses `pretty_midi` to form a piano-roll matrix.

In [ ]:
sample_path = midi_files[0]
print("Example MIDI:", sample_path)
try:
    score = converter.parse(str(sample_path), format="midi")
    note_objects = score.flatten().notes
    print("Note/chord objects:", len(note_objects))
    for element in list(note_objects)[:8]:
        print(type(element).__name__, str(element), "duration:", float(element.duration.quarterLength))
except Exception as exc:
    print("This example could not be parsed by music21; pretty_midi preprocessing will still be attempted.", exc)

## 5. Convert MIDI into polyphonic piano rolls

At each time step, a 60-value vector represents notes from MIDI pitches 36–95 (C2–B6). A value of `1` means the note is active. A **polyphonic** frame can contain several active notes, so the model can learn chords as well as melodies.

For mobile-friendly Colab training, we cap the amount read from each song and cap the total number of training windows. Increase these limits only if your Colab memory/runtime allows it.

In [ ]:
FPS = 8                         # Eight time steps per second
PITCH_MIN = 36                  # C2
PITCH_MAX = 96                  # exclusive; B6 is 95
PITCH_COUNT = PITCH_MAX - PITCH_MIN
SEQUENCE_LENGTH = 64            # 8 seconds of musical context at FPS=8
MAX_FILES_PER_GENRE = 900
MAX_SECONDS_PER_FILE = 120
WINDOW_STRIDE = 8
MAX_TRAIN_WINDOWS = 18000
MAX_VAL_WINDOWS = 3000
MAX_SEEDS_PER_GENRE = 24

GENRES = {
    "classical", "jazz", "pop", "rock", "electronic", "ambient", "cinematic",
    "orchestral", "acoustic", "indian_fusion", "indian_classical", "folk", "blues",
    "world_music", "lo_fi", "neo_soul", "fantasy",
}
GENRE_ALIASES = {"lofi": "lo_fi", "lo-fi": "lo_fi", "world": "world_music", "indian": "indian_classical"}

def normalize_folder(value: str) -> str:
    key = value.lower().strip().replace(" ", "_").replace("-", "_")
    return GENRE_ALIASES.get(key, key)

def genre_for_path(path: Path) -> str:
    try:
        relative_parts = path.relative_to(DATA_DIR).parts[:-1]
    except ValueError:
        relative_parts = path.parts[:-1]
    for part in relative_parts:
        key = normalize_folder(part)
        if key in GENRES:
            return key
    # MAESTRO is a classical solo-piano corpus; unlabelled MIDI defaults to Classical.
    return "classical"

def midi_to_roll(path: Path):
    midi = pretty_midi.PrettyMIDI(str(path))
    end_time = min(float(midi.get_end_time()), MAX_SECONDS_PER_FILE)
    if end_time <= 0.1:
        return None
    frames = max(SEQUENCE_LENGTH + 1, int(math.ceil(end_time * FPS)))
    roll = np.zeros((frames, PITCH_COUNT), dtype=np.uint8)
    for instrument in midi.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            pitch_index = int(note.pitch) - PITCH_MIN
            if not 0 <= pitch_index < PITCH_COUNT:
                continue
            start = max(0, min(frames, int(math.floor(note.start * FPS))))
            end = max(start + 1, min(frames, int(math.ceil(note.end * FPS))))
            roll[start:end, pitch_index] = 1
    # Skip files outside the represented pitch range or with too few note events.
    if int(roll.sum()) < 20:
        return None
    return roll

file_groups = defaultdict(list)
for path in midi_files:
    file_groups[genre_for_path(path)].append(path)

selected_paths = []
for genre_name, paths in sorted(file_groups.items()):
    random.shuffle(paths)
    selected_paths.extend((path, genre_name) for path in paths[:MAX_FILES_PER_GENRE])
random.shuffle(selected_paths)

records_by_genre = defaultdict(list)
failed_files = []
for index, (path, genre_name) in enumerate(selected_paths, 1):
    try:
        roll = midi_to_roll(path)
        if roll is not None and len(roll) > SEQUENCE_LENGTH:
            records_by_genre[genre_name].append((path, roll))
    except Exception as exc:
        failed_files.append((str(path), str(exc)[:120]))
    if index % 100 == 0:
        parsed = sum(len(rows) for rows in records_by_genre.values())
        print(f"Processed {index}/{len(selected_paths)} files; usable: {parsed}")

GENRE_NAMES = sorted(name for name, records in records_by_genre.items() if records)
if not GENRE_NAMES:
    raise RuntimeError("No usable MIDI files were processed. Verify that the dataset contains standard MIDI files.")
GENRE_INDEX = {name: i for i, name in enumerate(GENRE_NAMES)}
training_file_count = sum(len(rows) for rows in records_by_genre.values())
print("Genres represented in the data:", GENRE_NAMES)
print("Usable files:", training_file_count, "| Skipped files:", len(failed_files))
for name in GENRE_NAMES:
    print(f"  {name}: {len(records_by_genre[name])} files")

## 6. Build training and validation windows

Each input window contains the previous 64 polyphonic frames **plus a one-hot genre condition**. The target is the next 60-note frame. Files—not individual windows—are split into training and validation sets to reduce leakage between the same performance's overlapping windows.

In [ ]:
train_records, val_records = [], []
for genre_name in GENRE_NAMES:
    records = records_by_genre[genre_name][:]
    random.shuffle(records)
    if len(records) >= 10:
        n_val = max(1, int(round(len(records) * 0.10)))
    elif len(records) >= 2:
        n_val = 1
    else:
        n_val = 0
    val_records.extend((roll, genre_name) for _, roll in records[:n_val])
    train_records.extend((roll, genre_name) for _, roll in records[n_val:])


def collect_window_refs(records):
    refs = []
    for roll, genre_name in records:
        for start in range(0, len(roll) - SEQUENCE_LENGTH, WINDOW_STRIDE):
            refs.append((roll, start, GENRE_INDEX[genre_name]))
    random.shuffle(refs)
    return refs

train_refs = collect_window_refs(train_records)[:MAX_TRAIN_WINDOWS]
val_refs = collect_window_refs(val_records)[:MAX_VAL_WINDOWS]
if len(train_refs) < 100:
    raise RuntimeError(f"Only {len(train_refs)} training windows were made. Add more MIDI files or reduce SEQUENCE_LENGTH.")

INPUT_WIDTH = PITCH_COUNT + len(GENRE_NAMES)

def materialize_windows(refs):
    X = np.zeros((len(refs), SEQUENCE_LENGTH, INPUT_WIDTH), dtype=np.uint8)
    y = np.zeros((len(refs), PITCH_COUNT), dtype=np.uint8)
    for i, (roll, start, genre_index) in enumerate(refs):
        X[i, :, :PITCH_COUNT] = roll[start:start + SEQUENCE_LENGTH]
        X[i, :, PITCH_COUNT + genre_index] = 1
        y[i] = roll[start + SEQUENCE_LENGTH]
    return X, y

X_train, y_train = materialize_windows(train_refs)
X_val, y_val = materialize_windows(val_refs) if val_refs else (None, None)
print("Training input:", X_train.shape, "target:", y_train.shape)
if X_val is not None:
    print("Validation input:", X_val.shape, "target:", y_val.shape)
print("Input channels = note pitches + genre condition:", INPUT_WIDTH)

## 7. Train the LSTM from random initialization

This architecture uses two LSTM layers followed by a sigmoid output layer. The custom weighted binary-cross-entropy loss helps balance active note labels against the many inactive pitches in a piano roll. No pretrained weights are loaded. Use the validation loss to decide whether the model is learning; more epochs do not guarantee more musical output.

In [ ]:
BATCH_SIZE = 64
EPOCHS = 12
POSITIVE_NOTE_WEIGHT = 4.0


def make_dataset(X, y, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        dataset = dataset.shuffle(min(len(X), 8192), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE).map(
        lambda features, labels: (tf.cast(features, tf.float32), tf.cast(labels, tf.float32)),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return dataset.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val) if X_val is not None and len(X_val) else None


def weighted_binary_crossentropy(y_true, y_pred):
    eps = tf.keras.backend.epsilon()
    p = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    loss = -(POSITIVE_NOTE_WEIGHT * y_true * tf.math.log(p) + (1.0 - y_true) * tf.math.log(1.0 - p))
    return tf.reduce_mean(loss, axis=-1)

inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH, INPUT_WIDTH), name="note_sequence_and_genre")
x = tf.keras.layers.LSTM(64, return_sequences=True, name="lstm1")(inputs)
x = tf.keras.layers.Dropout(0.20, name="dropout1")(x)
x = tf.keras.layers.LSTM(64, name="lstm2")(x)
x = tf.keras.layers.Dropout(0.15, name="dropout2")(x)
outputs = tf.keras.layers.Dense(PITCH_COUNT, activation="sigmoid", name="output_layer")(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="HathorCustomLSTM")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=weighted_binary_crossentropy,
    metrics=[tf.keras.metrics.BinaryAccuracy(name="binary_accuracy", threshold=0.4)],
)
model.summary()

callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss" if val_ds is not None else "loss", patience=3, restore_best_weights=True)]
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
print("Training completed. This is the custom LSTM trained during this notebook run.")

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(history.history.get("loss", []), label="training loss")
if history.history.get("val_loss"):
    plt.plot(history.history.get("val_loss", []), label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Weighted binary cross-entropy")
plt.title("Hathor custom LSTM learning curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 8. Generate a new piano-roll sequence and export a sample MIDI

The LSTM uses a short seed context selected from the matching genre's training set, then predicts every output frame autoregressively. The emitted composition is the **newly predicted continuation**, not the copied seed. Tempo scales the MIDI timing; energy and complexity influence the sampling density.

In [ ]:
# Create small seed contexts per trained genre. These are only contexts for generation,
# not the whole training dataset and not pretrained model weights.
seed_rolls_list, seed_genre_indices_list = [], []
for genre_name in GENRE_NAMES:
    candidates = [roll for roll, name in train_records if name == genre_name and len(roll) > SEQUENCE_LENGTH]
    random.shuffle(candidates)
    for roll in candidates[:MAX_SEEDS_PER_GENRE]:
        max_start = max(0, len(roll) - SEQUENCE_LENGTH - 1)
        start = random.randint(0, max_start) if max_start else 0
        seed_rolls_list.append(roll[start:start + SEQUENCE_LENGTH].copy())
        seed_genre_indices_list.append(GENRE_INDEX[genre_name])
seed_rolls = np.asarray(seed_rolls_list, dtype=np.uint8)
seed_genre_indices = np.asarray(seed_genre_indices_list, dtype=np.int32)
print("Seed contexts:", seed_rolls.shape, "Genres:", GENRE_NAMES)


def sample_frame(probabilities, rng, mood="Dreamy", energy=3, complexity=3):
    mood_temperature = {"Dreamy":0.84,"Calm":0.76,"Epic":1.08,"Joyful":0.98,"Melancholic":0.86,"Mysterious":1.02,"Romantic":0.88,"Peaceful":0.74,"Dark":1.04,"Hopeful":0.94,"Spiritual":0.82,"Playful":1.05}.get(mood, 0.9)
    temperature = float(np.clip(mood_temperature + (complexity - 3) * 0.07 + (energy - 3) * 0.025, 0.55, 1.35))
    p = np.clip(probabilities.astype(np.float64), 1e-5, 1.0 - 1e-5)
    logits = np.log(p / (1.0 - p)) / temperature
    tuned = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    scores = np.log(np.clip(tuned, 1e-7, 1.0)) - np.log(np.clip(1.0 - tuned, 1e-7, 1.0))
    scores += rng.gumbel(0.0, 0.12, size=scores.shape)
    max_active = max(2, min(8, complexity + 2 + (1 if energy >= 4 else 0)))
    minimum = 0.16 if energy <= 2 else 0.12
    frame = np.zeros(PITCH_COUNT, dtype=np.uint8)
    for idx in np.argsort(scores)[::-1]:
        if tuned[idx] < minimum:
            break
        frame[idx] = 1
        if int(frame.sum()) >= max_active:
            break
    if not frame.any() and float(tuned.max()) > 0.075:
        frame[int(np.argmax(tuned))] = 1
    return frame


def model_predict_next(window, genre_name):
    onehot = np.zeros(len(GENRE_NAMES), dtype=np.uint8)
    onehot[GENRE_INDEX[genre_name]] = 1
    condition = np.broadcast_to(onehot, (SEQUENCE_LENGTH, len(GENRE_NAMES)))
    features = np.concatenate([window, condition], axis=1).astype(np.float32)
    return model(features[None, ...], training=False).numpy()[0]


def generate_sample_roll(genre_name="classical", seconds=20, tempo=100, energy=3, complexity=3, seed=123):
    eligible = np.flatnonzero(seed_genre_indices == GENRE_INDEX[genre_name])
    if not len(eligible):
        raise ValueError(f"No seed context for {genre_name}")
    rng = np.random.default_rng(seed)
    window = seed_rolls[int(rng.choice(eligible))].copy()
    n_frames = max(8, int(np.ceil(seconds * FPS * tempo / 120.0)))
    output = np.zeros((n_frames, PITCH_COUNT), dtype=np.uint8)
    for i in range(n_frames):
        probs = model_predict_next(window, genre_name)
        frame = sample_frame(probs, rng, energy=energy, complexity=complexity)
        output[i] = frame
        window = np.concatenate([window[1:], frame[None, :]], axis=0)
    # Fill one-frame holes and remove one-frame spikes.
    if len(output) > 2:
        for t in range(1, len(output) - 1):
            output[t, (output[t - 1] > 0) & (output[t + 1] > 0)] = 1
    for pitch in range(PITCH_COUNT):
        active = output[:, pitch].astype(bool)
        starts = np.flatnonzero(active & ~np.r_[False, active[:-1]])
        ends = np.flatnonzero(active & ~np.r_[active[1:], False]) + 1
        for start, end in zip(starts, ends):
            if end - start < 2:
                output[start:end, pitch] = 0
    return output


def roll_to_midi(roll, path, tempo=100, program=0, seconds=20):
    midi = pretty_midi.PrettyMIDI(initial_tempo=float(tempo), resolution=480)
    instrument = pretty_midi.Instrument(program=program, name="Hathor LSTM")
    frame_seconds = 120.0 / tempo / FPS
    for column in range(roll.shape[1]):
        active = roll[:, column].astype(bool)
        starts = np.flatnonzero(active & ~np.r_[False, active[:-1]])
        ends = np.flatnonzero(active & ~np.r_[active[1:], False]) + 1
        pitch = PITCH_MIN + column
        for start, end in zip(starts, ends):
            note_start = start * frame_seconds
            note_end = min(end * frame_seconds, seconds)
            if note_end - note_start >= 0.10:
                instrument.notes.append(pretty_midi.Note(velocity=78, pitch=int(pitch), start=float(note_start), end=float(note_end)))
    midi.instruments.append(instrument)
    midi.write(str(path))
    return midi

SAMPLE_GENRE = "classical" if "classical" in GENRE_NAMES else GENRE_NAMES[0]
SAMPLE_SECONDS, SAMPLE_TEMPO = 20, 100
sample_roll = generate_sample_roll(SAMPLE_GENRE, SAMPLE_SECONDS, SAMPLE_TEMPO, energy=3, complexity=3)
SAMPLE_MIDI = Path("/content/hathor_lstm_sample.mid")
sample_midi = roll_to_midi(sample_roll, SAMPLE_MIDI, tempo=SAMPLE_TEMPO, program=0, seconds=SAMPLE_SECONDS)
print("Sample MIDI saved:", SAMPLE_MIDI, "| Notes:", sum(len(i.notes) for i in sample_midi.instruments))

## 9. Export the trained model artifact for Render

This NPZ file contains the actual weights learned by this notebook, the list of training genres, and small seed contexts. The website's server uses **NumPy only** for inference; TensorFlow isn't needed in the Render container.

In [ ]:
MODEL_DIR = Path("/content/hathor_export")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
KERAS_PATH = MODEL_DIR / "hathor_lstm.keras"
WEIGHTS_PATH = MODEL_DIR / "hathor_lstm_weights.npz"
METADATA_PATH = MODEL_DIR / "hathor_lstm_metadata.json"

model.save(KERAS_PATH)
metadata = {
    "project": "Hathor AI",
    "model_type": "custom LSTM polyphonic piano-roll predictor",
    "training_method": "initialized from random weights and trained in this Colab notebook; no pretrained music-generation model used",
    "sequence_length": int(SEQUENCE_LENGTH),
    "pitch_count": int(PITCH_COUNT),
    "pitch_min": int(PITCH_MIN),
    "pitch_max_exclusive": int(PITCH_MAX),
    "frames_per_second": int(FPS),
    "genre_names": GENRE_NAMES,
    "genre_counts": {name: len(records_by_genre[name]) for name in GENRE_NAMES},
    "training_file_count": int(training_file_count),
    "training_windows": int(len(X_train)),
    "validation_windows": int(0 if X_val is None else len(X_val)),
    "seed_count": int(len(seed_rolls)),
    "architecture": {"lstm1_units": 64, "lstm2_units": 64, "output_pitch_count": PITCH_COUNT},
    "training_corpus": "MAESTRO MIDI plus optional user-supplied labelled MIDI folders",
    "default_dataset_license_note": "MAESTRO CC BY-NC-SA 4.0; use non-commercially unless you have confirmed other rights. Additional datasets retain their own licences.",
}

lstm1_weights = model.get_layer("lstm1").get_weights()
lstm2_weights = model.get_layer("lstm2").get_weights()
dense_weights = model.get_layer("output_layer").get_weights()
np.savez_compressed(
    WEIGHTS_PATH,
    lstm1_kernel=lstm1_weights[0],
    lstm1_recurrent_kernel=lstm1_weights[1],
    lstm1_bias=lstm1_weights[2],
    lstm2_kernel=lstm2_weights[0],
    lstm2_recurrent_kernel=lstm2_weights[1],
    lstm2_bias=lstm2_weights[2],
    dense_kernel=dense_weights[0],
    dense_bias=dense_weights[1],
    seed_rolls=seed_rolls,
    seed_genre_indices=seed_genre_indices,
    metadata_json=np.array(json.dumps(metadata)),
)
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Exported weights:", WEIGHTS_PATH, f"({WEIGHTS_PATH.stat().st_size / 1024:.1f} KB)")
print("Supported genres:", [name.replace("_", " ").title() for name in GENRE_NAMES])
print("Upload hathor_lstm_weights.npz to your GitHub repository at model/hathor_lstm_weights.npz")

## 10. Render a preview MP3 in Colab (optional but recommended)

FluidSynth is a conventional MIDI synthesizer that uses a SoundFont. It does **not** generate the musical notes—the custom LSTM has already generated the MIDI note sequence above.

In [ ]:
!apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm ffmpeg

SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
SAMPLE_WAV = Path("/content/hathor_lstm_sample.wav")
SAMPLE_MP3 = Path("/content/hathor_lstm_sample.mp3")
if Path(SOUNDFONT).is_file():
    subprocess.run([
        "fluidsynth", "-ni", "-g", "0.8", "-r", "44100", "-T", "wav", "-O", "s16",
        "-F", str(SAMPLE_WAV), SOUNDFONT, str(SAMPLE_MIDI),
    ], check=True, capture_output=True, text=True, timeout=120)
    subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(SAMPLE_WAV),
        "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-codec:a", "libmp3lame", "-b:a", "192k",
        "-ar", "44100", "-ac", "2", str(SAMPLE_MP3),
    ], check=True, capture_output=True, text=True, timeout=120)
    from IPython.display import Audio, display
    display(Audio(filename=str(SAMPLE_MP3)))
    print("Sample MP3:", SAMPLE_MP3)
else:
    print("SoundFont not found. The sample MIDI is still available; use FluidSynth locally or on Render to make audio.")

## 11. Download the model files to your Android phone

Run the cell below. Keep the files somewhere easy to find in your phone's Downloads folder. In GitHub/Acode, upload **only `hathor_lstm_weights.npz`** into your repository at `model/hathor_lstm_weights.npz` (make the `model` folder if needed). The `.keras` file and metadata JSON are useful research artifacts but are not required by Render.

In [ ]:
from google.colab import files
files.download(str(WEIGHTS_PATH))
files.download(str(METADATA_PATH))
files.download(str(SAMPLE_MIDI))
if SAMPLE_MP3.exists():
    files.download(str(SAMPLE_MP3))
# Optional: also download the full Keras training model for your research records.
# files.download(str(KERAS_PATH))

## 12. Connect it to the website

1. Add `hathor_lstm_weights.npz` to `CodeAlpha_HathorAI/model/` in GitHub.
2. Redeploy the Render Docker service.
3. Open `https://YOUR-SERVICE.onrender.com/api/model-info`. It should report `model_ready: true` and list only styles represented in the dataset.
4. Run a short track and check it by ear. If the output is repetitive or sparse, add more licensed MIDI training files, tune the training settings, and retrain.

**Do not claim this is a state-of-the-art audio model.** It is your own LSTM trained to generate symbolic MIDI note patterns. The sound character comes from the selected SoundFont instrument preset, and true text-to-music understanding, detailed dynamics, and production/mixing are outside this starter's model.